In [1]:
from llama_cpp import Llama
import time, hashlib

In [2]:
print("Loading model on CPU...")

llm = Llama(
    model_path="./models/phi-2.Q4_K_M.gguf",
    n_ctx=2048,        # context window
    n_threads=4,       # use 4 CPU cores — change to match your CPU
    verbose=False
)

print("✅ Model loaded!")

Loading model on CPU...
✅ Model loaded!


In [3]:
def measure_speed(prompt, max_tokens=100):
    start = time.time()
    output = llm(prompt, max_tokens=max_tokens, echo=False)
    elapsed = time.time() - start

    text = output["choices"][0]["text"]
    n_tokens = len(text.split())
    tps = n_tokens / elapsed

    print(f"Tokens/sec: {tps:.1f}")
    print(f"Time: {elapsed:.1f}s")
    return tps, text

In [4]:
PROMPT = "Explain how a transformer model works in simple terms."

print("Measuring speed...")
speed, output = measure_speed(PROMPT)
print("\nOUTPUT:")
print("-" * 40)
print(output)

Measuring speed...
Tokens/sec: 5.5
Time: 14.2s

OUTPUT:
----------------------------------------


Solution:

A transformer model is a type of neural network architecture used in natural language processing (NLP) tasks such as machine translation, text summarization, and sentiment analysis. It is based on the idea of self-attention, which allows the model to focus on different parts of the input sequence and weight their relevance to the output.

The basic idea behind a transformer is to represent each word in the input sequence as a vector in a high-dimensional space, and


In [5]:
results = {}

for threads in [1, 2, 4, 8]:
    llm_t = Llama(
        model_path="./models/phi-2.Q4_K_M.gguf",
        n_ctx=512,
        n_threads=threads,
        verbose=False
    )
    start = time.time()
    out = llm_t(PROMPT, max_tokens=50, echo=False)
    elapsed = time.time() - start
    tps = len(out["choices"][0]["text"].split()) / elapsed
    results[threads] = round(tps, 1)
    print(f"Threads: {threads} → {tps:.1f} tok/sec")

best_threads = max(results, key=results.get)
print(f"\nBest: {best_threads} threads at {results[best_threads]} tok/sec")

llama_context: n_ctx_seq (512) < n_ctx_train (2048) -- the full capacity of the model will not be utilized


Threads: 1 → 2.6 tok/sec


llama_context: n_ctx_seq (512) < n_ctx_train (2048) -- the full capacity of the model will not be utilized


Threads: 2 → 3.7 tok/sec


llama_context: n_ctx_seq (512) < n_ctx_train (2048) -- the full capacity of the model will not be utilized


Threads: 4 → 5.4 tok/sec


llama_context: n_ctx_seq (512) < n_ctx_train (2048) -- the full capacity of the model will not be utilized


Threads: 8 → 5.2 tok/sec

Best: 4 threads at 5.4 tok/sec


In [6]:
cache = {}

def cached_generate(prompt, max_tokens=100):
    key = hashlib.md5(prompt.strip().lower().encode()).hexdigest()
    if key in cache:
        print("⚡ CACHE HIT — instant response")
        return cache[key]
    print("🔄 Generating...")
    output = llm(prompt, max_tokens=max_tokens, echo=False)
    result = output["choices"][0]["text"]
    cache[key] = result
    return result

# First call
r1 = cached_generate("What is overfitting?")
print(r1[:150])

# Second call — instant
r2 = cached_generate("What is overfitting?")

🔄 Generating...

Answer: Overfitting is when a model learns too much detail from the training data and performs poorly on new, unseen data.

Exercise 3:
What is the d
⚡ CACHE HIT — instant response


In [7]:
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio, uvicorn, threading, time

nest_asyncio.apply()

app = FastAPI()

class Request(BaseModel):
    prompt: str
    max_tokens: int = 100

@app.post("/generate")
def generate(req: Request):
    start = time.time()
    output = llm(req.prompt, max_tokens=req.max_tokens, echo=False)
    text = output["choices"][0]["text"]
    latency_ms = round((time.time() - start) * 1000)
    return {"response": text, "latency_ms": latency_ms}

@app.get("/health")
def health():
    return {"status": "ok", "device": "cpu"}

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
print("✅ Server running at http://localhost:8000")

✅ Server running at http://localhost:8000


In [8]:
import requests

resp = requests.post("http://localhost:8000/generate", json={
    "prompt": "What is machine learning?",
    "max_tokens": 80
})
print(resp.json())

INFO:     Started server process [4080]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:53134 - "POST /generate HTTP/1.1" 200 OK
{'response': '\nAnswer: Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance without being explicitly programmed.\n\nExercise 2:\nWhat is a neural network?\nAnswer: A neural network is a type of machine learning model that is designed to simulate the way that neurons in the brain work.\n\nExercise 3:\nWhat is sigmoid node', 'latency_ms': 10640}


In [9]:
import gradio as gr

def respond(prompt, max_tokens):
    return cached_generate(prompt, int(max_tokens))

gr.Interface(
    fn=respond,
    inputs=[
        gr.Textbox(lines=3, placeholder="Ask anything...", label="Prompt"),
        gr.Slider(20, 200, value=80, step=10, label="Max Tokens"),
    ],
    outputs=gr.Textbox(lines=5, label="Response"),
    title="Phi-2 CPU Inference",
    examples=[
        ["What is gradient descent?", 80],
        ["Explain overfitting simply.", 80],
    ]
).launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://89f2f8f5d6e0538a93.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🔄 Generating...
